# Bài 8.4 · Ensemble: gộp nhiều model

**Câu hỏi của buổi này:** gộp dự đoán của nhiều model được thêm bao nhiêu điểm, khi nào gần như không được gì, và làm sao để con số báo cáo không đẹp giả?

Dữ liệu là **Bank Marketing** (UCI): 41 188 cuộc gọi chào gửi tiết kiệm của một ngân hàng Bồ Đào Nha, đoán khách có đồng ý không (11% đồng ý). Chấm bằng **AUC**. Cột `duration` (độ dài cuộc gọi) bị bỏ ngay từ đầu: chỉ biết sau khi gọi xong, tức là rò rỉ (bài 8.2).

Dữ liệu chia ba phần có vai trò riêng: **train** (60%) để học, **blend** (20%) để chọn cách gộp, **test** (20%) chỉ dùng để chấm lần cuối.

| Mục | Sẽ thấy |
|---|---|
| 1. Năm model | AUC từng model trên blend và test, thời gian chạy |
| 2. Bỏ phiếu và trung bình | gộp nhãn khác gộp xác suất |
| 3. Các model sai giống nhau đến đâu | ma trận tương quan |
| 4. Chọn tổ hợp trên blend, chấm trên test | mọi tổ hợp, điểm thật và điểm đẹp giả |
| 5. Trọng số | lưới trọng số chọn trên blend, so với chọn trên test |
| 6. Hơn một chút có thật không | bootstrap trên test: khoảng tin cậy của phần hơn |
| 7. Trung bình nhiều seed | AUC theo số seed của MLP |
| 8. Stacking | out-of-fold đúng cách và cách làm sai |
| 9. Ghép sai thứ tự | một model lệch hàng, cả ensemble sụp |
| 10. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Chỉ cần CPU, bật **Internet** trong *Session options*; chạy hết khoảng 2 phút. Bài giảng: [ai-canuc80k · bài 8.4](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/8.4).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import io, time, zipfile, itertools, urllib.request, warnings
from pathlib import Path
import sklearn
from scipy.stats import rankdata, spearmanr
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
warnings.filterwarnings('ignore', category=UserWarning)

root = Path('/tmp/bank'); root.mkdir(parents=True, exist_ok=True)
if not (root / 'data.zip').exists():
    urllib.request.urlretrieve('https://archive.ics.uci.edu/static/public/222/bank+marketing.zip', root / 'data.zip')
inner = zipfile.ZipFile(io.BytesIO(zipfile.ZipFile(root / 'data.zip').read('bank-additional.zip')))
d = pd.read_csv(inner.open('bank-additional/bank-additional-full.csv'), sep=';').drop(columns='duration')
y = (d.pop('y') == 'yes').astype(int).values
cat = d.select_dtypes('object').columns.tolist(); num = [c for c in d if c not in cat]

idx = np.arange(len(y))
trv, te = train_test_split(idx, test_size=.2, random_state=0, stratify=y)
tr, bl = train_test_split(trv, test_size=.25, random_state=0, stratify=y[trv])
table(pd.DataFrame([('train', len(tr), y[tr].mean()), ('blend', len(bl), y[bl].mean()), ('test', len(te), y[te].mean())],
                   columns=['phần', 'số dòng', 'tỉ lệ đồng ý']), f'Bank Marketing: {len(y)} cuộc gọi, {d.shape[1]} cột (scikit-learn {sklearn.__version__})', fmt={'tỉ lệ đồng ý': '{:.1%}'})
check('ba phần không chồng nhau và đủ mọi dòng', len(set(tr) | set(bl) | set(te)) == len(y) == len(tr) + len(bl) + len(te))

## 1. Năm model khác kiểu

Logistic regression và kNN trên dữ liệu one-hot đã chuẩn hóa; random forest và gradient boosting trên mã số của cột chữ; một MLP nhỏ. Mỗi model học trên train, dự đoán xác suất cho blend và test.

In [ ]:
def onehot(): return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), cat), ('n', StandardScaler(), num)])
def ordinal(): return ColumnTransformer([('c', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat)], remainder='passthrough')
MODELS = {
    'logistic': lambda s: make_pipeline(onehot(), LogisticRegression(max_iter=2000, C=.5)),
    'kNN': lambda s: make_pipeline(onehot(), KNeighborsClassifier(100)),
    'random forest': lambda s: make_pipeline(ordinal(), RandomForestClassifier(300, min_samples_leaf=5, n_jobs=-1, random_state=s)),
    'gradient boosting': lambda s: make_pipeline(ordinal(), HistGradientBoostingClassifier(learning_rate=.08, max_iter=150, random_state=s)),
    'MLP': lambda s: make_pipeline(onehot(), MLPClassifier((64,), alpha=1e-3, max_iter=200, early_stopping=True, random_state=s)),
}
NAMES = list(MODELS)
PB, PT, cost = {}, {}, []
for n, mk in MODELS.items():
    t = time.time(); m = mk(0).fit(d.iloc[tr], y[tr]); t_fit = time.time() - t
    t = time.time(); PB[n] = m.predict_proba(d.iloc[bl])[:, 1]; PT[n] = m.predict_proba(d.iloc[te])[:, 1]; t_pred = time.time() - t
    cost.append((n, roc_auc_score(y[bl], PB[n]), roc_auc_score(y[te], PT[n]), t_fit, t_pred))
single = pd.DataFrame(cost, columns=['model', 'AUC blend', 'AUC test', 'giây train', 'giây dự đoán'])
table(single, 'Từng model', fmt={'AUC blend': '{:.4f}', 'AUC test': '{:.4f}', 'giây train': '{:.1f}', 'giây dự đoán': '{:.1f}'}, best='AUC blend')
best_single = single.loc[single['AUC blend'].idxmax(), 'model']
best_test = single.set_index('model').loc[best_single, 'AUC test']
check('mọi model hơn đoán ngẫu nhiên rõ rệt', (single['AUC test'] > 0.7).all())
note(f'Model tốt nhất trên blend là <b>{best_single}</b> (test {best_test:.4f}). Năm model cách nhau chưa tới {single["AUC test"].max() - single["AUC test"].min():.3f} AUC trên test. '
     f'Đây là mốc mà mọi cách gộp phía dưới phải vượt.')

## 2. Gộp nhãn khác gộp xác suất

**Bỏ phiếu**: mỗi model nói có/không ở ngưỡng 0.5, theo đa số. **Trung bình xác suất**: cộng xác suất rồi chia. Với AUC, chỉ thứ tự quan trọng nên còn cách thứ ba, **trung bình hạng**: đổi mỗi cột dự đoán thành hạng rồi lấy trung bình.

In [ ]:
T = np.column_stack([PT[n] for n in NAMES]); Bm = np.column_stack([PB[n] for n in NAMES])
vote = (T > .5).sum(1) >= 3
mean_p = T.mean(1)
rank_mean = np.column_stack([rankdata(c) for c in T.T]).mean(1)
rows = [('bỏ phiếu đa số (nhãn)', roc_auc_score(y[te], (T > .5).mean(1)), accuracy_score(y[te], vote), f1_score(y[te], vote)),
        ('trung bình xác suất', roc_auc_score(y[te], mean_p), accuracy_score(y[te], mean_p > .5), f1_score(y[te], mean_p > .5)),
        ('trung bình hạng', roc_auc_score(y[te], rank_mean), np.nan, np.nan)]
g = pd.DataFrame(rows, columns=['cách gộp năm model', 'AUC test', 'accuracy (ngưỡng 0.5)', 'F1 (ngưỡng 0.5)'])
table(g, 'Ba cách gộp, cùng năm model', fmt={c: '{:.4f}' for c in g.columns[1:]})
levels = len(np.unique((T > .5).mean(1)))
check('bỏ phiếu chỉ cho vài mức điểm, trung bình xác suất cho rất nhiều mức', levels <= 6 and len(np.unique(mean_p)) > 1000)
note(f'Bỏ phiếu biến mỗi khách thành một trong {levels} mức (0/5 … 5/5 model nói có), nên AUC chỉ {g.iloc[0, 1]:.3f}: thứ tự bên trong mỗi mức mất hết. '
     f'Trung bình xác suất giữ thứ tự, AUC {g.iloc[1, 1]:.4f}; trung bình hạng {g.iloc[2, 1]:.4f}. '
     f'Gộp cả năm model {"chưa vượt" if g.iloc[1, 1] <= best_test else "vượt"} model tốt nhất ({best_test:.4f}): một model yếu trong nhóm có thể kéo điểm xuống.')

## 3. Các model sai giống nhau đến đâu

Gộp chỉ có lợi khi các model sai ở những chỗ khác nhau. Đo tương quan hạng (Spearman) giữa dự đoán của từng cặp trên test.

In [ ]:
corr = pd.DataFrame([[spearmanr(PT[a], PT[b])[0] for b in NAMES] for a in NAMES], index=NAMES, columns=NAMES)
fig, ax = plt.subplots(figsize=(6.6, 5.4))
im = ax.imshow(corr.values, cmap='magma', vmin=.6, vmax=1)
for i in range(5):
    for j in range(5): ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', color='white' if corr.values[i, j] < .9 else 'black', fontsize=10)
ax.set_xticks(range(5), NAMES, rotation=30, ha='right'); ax.set_yticks(range(5), NAMES); ax.grid(False)
ax.set_title('Tương quan hạng giữa dự đoán trên test'); fig.colorbar(im, fraction=.046)
plt.tight_layout(); plt.show()
off = corr.values[~np.eye(5, dtype=bool)]
pair_lo = min(((a, b) for a in NAMES for b in NAMES if a < b), key=lambda p: corr.loc[p])
note(f'Tương quan giữa các cặp từ {off.min():.2f} tới {off.max():.2f}. '
     + ('Năm model gần như xếp khách theo cùng một thứ tự, nên gộp lại khó thêm nhiều. ' if off.min() > .85 else 'Nhiều cặp xếp khách khá giống nhau, nên đừng mong gộp thêm nhiều điểm. ')
     + f'Cặp khác nhau nhất là {pair_lo[0]} và {pair_lo[1]} ({corr.loc[pair_lo]:.2f}): khác nhau là điều kiện cần để bù cho nhau, nhưng chưa đủ nếu một trong hai yếu hơn hẳn; mục 4 đo điều đó.')

## 4. Chọn tổ hợp trên blend, chấm trên test

Thử mọi tổ hợp từ 2 đến 5 model (26 tổ hợp), lấy trung bình xác suất. Chọn tổ hợp tốt nhất **theo blend**, rồi mới xem test. Nếu chọn theo test thì con số báo cáo đẹp hơn sự thật, vì đã dùng test để chọn.

In [ ]:
combos = [c for k in range(2, 6) for c in itertools.combinations(NAMES, k)]
cb = pd.DataFrame([(' + '.join(c), len(c), roc_auc_score(y[bl], np.mean([PB[n] for n in c], 0)), roc_auc_score(y[te], np.mean([PT[n] for n in c], 0))) for c in combos],
                  columns=['tổ hợp', 'số model', 'AUC blend', 'AUC test'])
pick = cb.loc[cb['AUC blend'].idxmax()]; cheat = cb.loc[cb['AUC test'].idxmax()]
fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(cb['AUC blend'], cb['AUC test'], c=cb['số model'], cmap='viridis', s=40)
ax.scatter(single['AUC blend'], single['AUC test'], marker='s', color=C['gray'], s=40, label='một model')
ax.scatter(pick['AUC blend'], pick['AUC test'], s=220, facecolors='none', edgecolors=C['clay'], lw=2, label='chọn theo blend')
ax.axhline(best_test, color=C['gray'], ls='--', lw=1, label=f'model tốt nhất (test {best_test:.4f})')
ax.set(xlabel='AUC trên blend (dùng để chọn)', ylabel='AUC trên test (chấm lần cuối)', title='Mỗi chấm một cách gộp'); ax.legend(frameon=False, loc='lower right')
fig.colorbar(sc, ax=ax, label='số model', fraction=.04)
plt.tight_layout(); plt.show()
table(cb.sort_values('AUC blend', ascending=False).head(5), 'Năm tổ hợp đứng đầu theo blend', fmt={'AUC blend': '{:.4f}', 'AUC test': '{:.4f}'})
check('blend và test xếp các tổ hợp gần giống nhau', spearmanr(cb['AUC blend'], cb['AUC test'])[0] > .7)
note(f'Chọn theo blend được <b>{pick["tổ hợp"]}</b>: test {pick["AUC test"]:.4f}, '
     + (f'hơn model tốt nhất {pick["AUC test"] - best_test:+.4f}. ' if pick['AUC test'] > best_test else f'không hơn model tốt nhất ({pick["AUC test"] - best_test:+.4f}). ')
     + (f'Nếu chọn thẳng theo test thì sẽ báo {cheat["AUC test"]:.4f} ({cheat["tổ hợp"]}): phần chênh {cheat["AUC test"] - pick["AUC test"]:+.4f} là điểm đẹp giả, không có lúc thi thật.'
        if cheat['tổ hợp'] != pick['tổ hợp'] else 'Lần này tổ hợp tốt nhất theo blend cũng là tốt nhất theo test: blend và test đủ lớn (mỗi phần hơn 8 000 dòng) để xếp hạng giống nhau.'))

## 5. Trung bình có trọng số

Thay trung bình đều bằng trọng số, gộp trên hạng như mục 2 (AUC chỉ xét thứ tự), quét mọi bộ trọng số bước 0.1 có tổng bằng 1 (1 001 bộ cho năm model). Lại chọn trên blend, chấm trên test.

In [ ]:
grid = [w for w in itertools.product(range(11), repeat=5) if sum(w) == 10]
W = np.array(grid) / 10
RB = np.column_stack([rankdata(c) for c in Bm.T]) / len(bl); RT = np.column_stack([rankdata(c) for c in T.T]) / len(te)
auc_b = np.array([roc_auc_score(y[bl], RB @ w) for w in W]); auc_t = np.array([roc_auc_score(y[te], RT @ w) for w in W])
w_pick, w_cheat = W[auc_b.argmax()], W[auc_t.argmax()]
wt = pd.DataFrame([('chọn trên blend', *w_pick, auc_b.max(), auc_t[auc_b.argmax()]), ('chọn trên test (gian lận)', *w_cheat, auc_b[auc_t.argmax()], auc_t.max())],
                  columns=['trọng số', *NAMES, 'AUC blend', 'AUC test'])
table(wt, f'Trung bình hạng có trọng số, {len(W)} bộ trọng số', fmt={**{n: '{:.1f}' for n in NAMES}, 'AUC blend': '{:.4f}', 'AUC test': '{:.4f}'})
check('trọng số mỗi dòng cộng lại bằng 1', np.allclose(W.sum(1), 1))
note(f'Trọng số chọn trên blend cho test {auc_t[auc_b.argmax()]:.4f}, '
     + ('nhỉnh hơn trung bình đều của tổ hợp tốt nhất ở mục 4. ' if auc_t[auc_b.argmax()] > pick['AUC test'] else 'không hơn trung bình đều của tổ hợp tốt nhất ở mục 4: thêm 4 tham số để chỉnh chỉ là thêm chỗ khớp nhiễu của blend. ')
     + f'Bộ trọng số "tốt nhất trên test" báo {auc_t.max():.4f}, hơn bộ chọn đúng cách {auc_t.max() - auc_t[auc_b.argmax()]:+.4f}. '
     + ('Phần hơn ấy rất nhỏ vì test lớn; test nhỏ thì nhiễu lớn hơn nên phần đẹp giả có thể lớn hơn nhiều (thử ở cuối notebook).' if auc_t.max() - auc_t[auc_b.argmax()] < .002 else 'Phần hơn ấy là điểm đẹp giả: lúc thi thật không có nhãn test để chọn.'))

## 6. Hơn một chút có thật không

Các cách gộp chỉ hơn model đơn cỡ 0.001 AUC. Lấy lại mẫu có hoàn lại từ test (bootstrap) 500 lần, mỗi lần tính phần hơn của trung bình có trọng số (chọn trên blend) so với gradient boosting, để xem phần hơn ấy có vượt dao động ngẫu nhiên không.

In [ ]:
rng = np.random.default_rng(0)
ens_t = RT @ w_pick; base_t = PT[best_single]
diffs = []
for _ in range(500):
    b = rng.integers(0, len(te), len(te))
    if y[te][b].min() == y[te][b].max(): continue
    diffs.append(roc_auc_score(y[te][b], ens_t[b]) - roc_auc_score(y[te][b], base_t[b]))
diffs = np.array(diffs); lo, hi = np.percentile(diffs, [2.5, 97.5])
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.hist(diffs, bins=40, color=C['blue'], alpha=.85); ax.axvline(0, color=C['clay'], lw=2)
ax.axvspan(lo, hi, color=C['sand'], alpha=.2, label=f'95% bootstrap: [{lo:+.4f}, {hi:+.4f}]')
ax.set(xlabel='AUC ensemble − AUC ' + best_single, ylabel='số lần', title=f'{len(diffs)} mẫu bootstrap của test'); ax.legend(frameon=False)
plt.tight_layout(); plt.show()
check('đủ mẫu bootstrap', len(diffs) > 450)
note(f'Phần hơn đo được trên test là {auc_t[auc_b.argmax()] - best_test:+.4f}; trong {(diffs > 0).mean():.0%} số mẫu bootstrap ensemble hơn. '
     + ('Khoảng 95% nằm hẳn trên 0: phần hơn nhỏ nhưng ổn định.' if lo > 0 else 'Khoảng 95% còn chứa 0: với test này chưa đủ chắc ensemble hơn hẳn model đơn.'))

## 7. Trung bình nhiều seed

Một nguồn khác nhau giữa các model là **ngẫu nhiên lúc train**: MLP khởi tạo khác thì ra trọng số khác. Train MLP với 8 seed, lấy trung bình 1, 2, 4, 8 bản.

In [ ]:
SEEDS = 8
t = time.time()
ms = [MODELS['MLP'](s).fit(d.iloc[tr], y[tr]).predict_proba(d.iloc[te])[:, 1] for s in range(SEEDS)]
t_seed = time.time() - t
each = [roc_auc_score(y[te], m) for m in ms]
ks = [k for k in (1, 2, 4, 8) if k <= SEEDS]
avg_k = [roc_auc_score(y[te], np.mean(ms[:k], 0)) for k in ks]
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.scatter([1] * SEEDS, each, color=C['gray'], s=25, label='từng seed')
ax.plot(ks, avg_k, 'o-', color=C['clay'], lw=2, label='trung bình k seed đầu')
for k, v in zip(ks, avg_k): ax.annotate(f'{v:.4f}', (k, v), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
ax.set_xscale('log', base=2); ax.set_xticks(ks, ks)
ax.set(xlabel='số seed lấy trung bình', ylabel='AUC test', title=f'MLP, {SEEDS} seed ({t_seed:.0f} giây)'); ax.legend(frameon=False, loc='lower right')
plt.tight_layout(); plt.show()
check('trung bình mọi seed không thua seed trung bình', avg_k[-1] >= np.mean(each))
note(f'Từng seed cho AUC từ {min(each):.4f} tới {max(each):.4f}; trung bình {ks[-1]} seed được {avg_k[-1]:.4f}, hơn mức trung bình của một seed ({np.mean(each):.4f}) {avg_k[-1] - np.mean(each):+.4f}. '
     f'Trung bình seed giảm phần dao động do ngẫu nhiên, không sửa được thiên lệch chung của MLP, nên vẫn {"thua" if avg_k[-1] < best_test else "hơn"} {best_single}. Chi phí: gấp {ks[-1]} lần thời gian train.')

## 8. Stacking: dự đoán out-of-fold

Stacking học một model nhỏ (meta-model) nhận dự đoán của các model gốc làm đầu vào. Đầu vào ấy phải là **dự đoán out-of-fold**: mỗi dòng của train+blend được dự đoán bởi một model **không** học dòng đó (3-fold). Cách làm sai là đưa cho meta-model dự đoán trên chính dữ liệu model gốc đã học.

In [ ]:
X_all = d.iloc[trv]; y_all = y[trv]
skf = StratifiedKFold(3, shuffle=True, random_state=0)
t = time.time()
oof = np.zeros((len(trv), 5)); test_from_full = np.zeros((len(te), 5)); insample = np.zeros((len(trv), 5))
for j, n in enumerate(NAMES):
    for a, b in skf.split(X_all, y_all):
        oof[b, j] = MODELS[n](0).fit(X_all.iloc[a], y_all[a]).predict_proba(X_all.iloc[b])[:, 1]
    full = MODELS[n](0).fit(X_all, y_all)
    insample[:, j] = full.predict_proba(X_all)[:, 1]; test_from_full[:, j] = full.predict_proba(d.iloc[te])[:, 1]
t_stack = time.time() - t
meta_ok = LogisticRegression(max_iter=1000).fit(oof, y_all)
meta_bad = LogisticRegression(max_iter=1000).fit(insample, y_all)
st = pd.DataFrame([('meta trên dự đoán out-of-fold', *meta_ok.coef_[0], roc_auc_score(y_all, meta_ok.predict_proba(oof)[:, 1]), roc_auc_score(y[te], meta_ok.predict_proba(test_from_full)[:, 1])),
                   ('meta trên dự đoán chính dữ liệu đã học (sai)', *meta_bad.coef_[0], roc_auc_score(y_all, meta_bad.predict_proba(insample)[:, 1]), roc_auc_score(y[te], meta_bad.predict_proba(test_from_full)[:, 1])),
                   ('trung bình đều năm model (học trên train+blend)', *[np.nan] * 5, np.nan, roc_auc_score(y[te], test_from_full.mean(1)))],
                  columns=['cách', *[f'hệ số {n}' for n in NAMES], 'AUC lúc học meta', 'AUC test'])
table(st, f'Stacking với meta-model logistic ({t_stack:.0f} giây cho 3 fold × 5 model)', fmt={c: '{:.2f}' for c in st.columns[1:6]} | {'AUC lúc học meta': '{:.4f}', 'AUC test': '{:.4f}'})
ins_auc = pd.Series([roc_auc_score(y_all, insample[:, j]) for j in range(5)], index=NAMES)
check('dự đoán trên chính dữ liệu đã học đẹp hơn out-of-fold', st.iloc[1, 6] > st.iloc[0, 6])
note(f'Trên dữ liệu đã học, random forest đạt AUC {ins_auc["random forest"]:.3f} (nó gần như thuộc lòng), nên meta-model sai tin nó: hệ số {st.iloc[1, 3]:.2f} so với {st.iloc[0, 3]:.2f} khi học đúng cách. '
     f'Kết quả trên test: đúng cách {st.iloc[0, 7]:.4f}, sai cách {st.iloc[1, 7]:.4f}. '
     f'AUC "lúc học meta" của cách sai ({st.iloc[1, 6]:.4f}) là con số đẹp giả điển hình.')

## 9. Ghép sai thứ tự

Ensemble chỉ đúng khi dòng thứ i của mọi file dự đoán là cùng một khách. Giả sử file của một model bị sắp lại theo ID khác thứ tự (ở đây: đảo ngẫu nhiên dự đoán của gradient boosting).

In [ ]:
rng = np.random.default_rng(0)
shuffled = T.copy(); shuffled[:, NAMES.index('gradient boosting')] = rng.permutation(shuffled[:, NAMES.index('gradient boosting')])
mis = pd.DataFrame([('năm model, đúng thứ tự', roc_auc_score(y[te], T.mean(1))), ('năm model, gradient boosting lệch hàng', roc_auc_score(y[te], shuffled.mean(1)))],
                   columns=['ghép', 'AUC test'])
table(mis, 'Cùng các dự đoán, chỉ khác thứ tự một cột', fmt={'AUC test': '{:.4f}'})
ids = pd.DataFrame({'id': te, 'p': PT['gradient boosting']}).sample(frac=1, random_state=0)
fixed = pd.DataFrame({'id': te}).merge(ids, on='id', how='left')['p'].values
check('ghép lại theo cột id cho đúng dự đoán ban đầu', np.allclose(fixed, PT['gradient boosting']))
note(f'Một cột lệch hàng kéo AUC từ {mis.iloc[0, 1]:.4f} xuống {mis.iloc[1, 1]:.4f}, mà không có lỗi nào được báo. Luôn ghép các file dự đoán bằng <code>merge</code> theo cột ID, '
     'rồi kiểm số dòng và vài ID ở đầu, cuối trước khi lấy trung bình.')

## 10. Tóm tắt

In [ ]:
summary = [('model đơn tốt nhất (chọn theo blend): AUC test', f'{best_single}: {best_test:.4f}'),
           ('tương quan hạng giữa các model', f'{off.min():.2f}–{off.max():.2f}'),
           ('bỏ phiếu / trung bình xác suất năm model: AUC test', f'{g.iloc[0, 1]:.4f} / {g.iloc[1, 1]:.4f}'),
           ('tổ hợp chọn theo blend: AUC test', f'{pick["AUC test"]:.4f} ({pick["tổ hợp"]})'),
           ('trọng số chọn trên blend / chọn trên test (đẹp giả): AUC test', f'{auc_t[auc_b.argmax()]:.4f} / {auc_t.max():.4f}'),
           ('phần hơn của ensemble so với model đơn: 95% bootstrap', f'[{lo:+.4f}, {hi:+.4f}]'),
           (f'MLP: một seed (trung bình) / trung bình {ks[-1]} seed', f'{np.mean(each):.4f} / {avg_k[-1]:.4f}'),
           ('stacking out-of-fold / sai cách: AUC test', f'{st.iloc[0, 7]:.4f} / {st.iloc[1, 7]:.4f}'),
           ('một cột lệch hàng: AUC test', f'{mis.iloc[1, 1]:.4f}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** giảm blend và test còn 800 dòng mỗi phần rồi xem chọn trên test đẹp giả tới đâu; giảm train còn 2 000 dòng rồi chạy lại mục 4 (model dao động nhiều hơn thì gộp có lợi hơn không?); thay meta-model logistic bằng gradient boosting ở mục 7; ở mục 6 trung bình seed cho random forest.